In [1]:
import os
from pathlib import Path
from src.utils import *
from src.config import PROCESSED_IMG_DIR, RAW_IMG_DIR
from src.feature_extraction import *
from src.feature_extraction import _create_fixed_box
from src.region_visualization import collect_all_rois, build_mosaic, show_mosaic
import time
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from sklearn.feature_selection import f_classif
from mpl_toolkits.mplot3d import Axes3D
import pandas as pd
from src.config import ROOT_DIR
import time
import cv2
import random
from multiprocessing import Pool, cpu_count
from functools import partial
from src.feature_extraction import (
    process_one_image_oriented, extract_oriented_rois,
    label_oriented_rois, crop_oriented_roi,
    build_gabor_bank, build_oriented_feature_column_names,
    _order_skeleton_points_fast
)

In [2]:
paths = get_all_image_paths(directory = PROCESSED_IMG_DIR)
separated_paths = seprate_processed_files(paths) # Separate all paths that end in .png
processed_paths = separated_paths[0] # These are the processed images (paths only)
masks_paths = separated_paths[1]     # These are the binary masks (paths only)
skeletons_paths = separated_paths[2] # These are the skeletons (paths only)
xml_paths = []                       # And this for the .xml (paths only)
ground_truth_bboxes = []             # And this for the ground truth bboxes
for img_path in processed_paths:
    xml_path = get_xml_path(img_path)
    xml_path = xml_path.replace('_processed.xml', '_bbox.xml') # Fix the ending from _processed.xml to _bbox.xml
    xml_paths.append(xml_path)
    bbox = parse_stenosis_xml(xml_path)
    ground_truth_bboxes.append(bbox)
print('XML files: ', len(xml_paths))
print('Ground truth bounding boxes: ', len(ground_truth_bboxes))

TOTAL_ROIS  = 100
RECT_WIDTH  = 100
RECT_HEIGHT = 50
N_SIZES        = 6
N_ORIENTATIONS = 12

OUTPUT_DIR_CSV = Path(os.path.join(ROOT_DIR, "notebooks/csv_files/oriented_rois"))

full_dataset = list(zip(processed_paths, masks_paths, skeletons_paths, ground_truth_bboxes))

Total images found: 24975
Processed images: 8325, Masks: 8325, Skeletons: 8325
XML files:  8325
Ground truth bounding boxes:  8325


## Extract only coordinates of the oriented ROIs

In [3]:
coord_worker_args = [
    (img_path, mask_path, skel_path, gt_boxes,
     TOTAL_ROIS, RECT_WIDTH, RECT_HEIGHT)
    for img_path, mask_path, skel_path, gt_boxes in full_dataset
]
# Test a single image with NO pool — pure serial
test_args = coord_worker_args[0]
t0 = time.time()
result = process_one_image_coords_oriented(test_args)
print(f"Single image: {time.time()-t0:.2f}s, ROIs returned: {len(result)}")

Single image: 0.23s, ROIs returned: 101


In [4]:
# ─────────────────────────────────────────────────────────────
# CELL A: Extract oriented ROI coordinates ONLY (no features)
#         → roi_oriented_coordinates.csv
# ─────────────────────────────────────────────────────────────
from src.feature_extraction import process_one_image_coords_oriented
from joblib import Parallel, delayed
from tqdm import tqdm

coord_worker_args = [
    (img_path, mask_path, skel_path, gt_boxes,
     TOTAL_ROIS, RECT_WIDTH, RECT_HEIGHT)
    for img_path, mask_path, skel_path, gt_boxes in full_dataset
]

n_cores = max(1, cpu_count() - 2)
print(f"CPUs available      : {cpu_count()}")
print(f"Workers spawned     : {n_cores}")
print(f"Tasks               : {len(coord_worker_args)}")
print(f"Extracting coordinates for {len(full_dataset)} images using {n_cores} cores...")

start = time.time()
coord_results = Parallel(n_jobs=n_cores, verbose=10)(
    delayed(process_one_image_coords_oriented)(args)
    for args in coord_worker_args
)
elapsed = time.time() - start
print(f"Done in {elapsed:.1f}s")

coord_rows = [row for image_rows in coord_results for row in image_rows]
df_coords  = pd.DataFrame(coord_rows)

# ── Sort by image order then ROI index ──────────────────────
df_coords[['_image_name', '_roi_idx']] = df_coords['roi_name'].str.rsplit('_', n=1, expand=True)
df_coords['_roi_idx'] = df_coords['_roi_idx'].astype(int)
df_coords = df_coords.sort_values(['_image_name', '_roi_idx']).drop(columns=['_image_name', '_roi_idx']).reset_index(drop=True)
# ────────────────────────────────────────────────────────────

print(f"Total ROIs       : {len(coord_rows)}")
print(f"Stenosis    (1)  : {(df_coords['label'] ==  1).sum()}")
print(f"Undetermined(-1) : {(df_coords['label'] == -1).sum()}")
print(f"Healthy     (0)  : {(df_coords['label'] ==  0).sum()}")

OUTPUT_DIR_CSV.mkdir(parents=True, exist_ok=True)
coords_path = OUTPUT_DIR_CSV / "roi_oriented_coordinates.csv"
df_coords.to_csv(coords_path, index=False)
print(f"Saved: {coords_path}")

CPUs available      : 12
Workers spawned     : 10
Tasks               : 8325
Extracting coordinates for 8325 images using 10 cores...


[Parallel(n_jobs=10)]: Using backend LokyBackend with 10 concurrent workers.
[Parallel(n_jobs=10)]: Done   5 tasks      | elapsed:    2.9s
[Parallel(n_jobs=10)]: Done  12 tasks      | elapsed:    3.1s
[Parallel(n_jobs=10)]: Done  21 tasks      | elapsed:    3.3s
[Parallel(n_jobs=10)]: Done  30 tasks      | elapsed:    3.6s
[Parallel(n_jobs=10)]: Done  41 tasks      | elapsed:    4.1s
[Parallel(n_jobs=10)]: Done  52 tasks      | elapsed:    4.7s
[Parallel(n_jobs=10)]: Done  65 tasks      | elapsed:    5.2s
[Parallel(n_jobs=10)]: Done  78 tasks      | elapsed:    6.0s
[Parallel(n_jobs=10)]: Done  93 tasks      | elapsed:    6.5s
[Parallel(n_jobs=10)]: Done 108 tasks      | elapsed:    8.1s
[Parallel(n_jobs=10)]: Done 125 tasks      | elapsed:    9.5s
[Parallel(n_jobs=10)]: Done 142 tasks      | elapsed:   10.7s
[Parallel(n_jobs=10)]: Done 161 tasks      | elapsed:   12.4s
[Parallel(n_jobs=10)]: Done 180 tasks      | elapsed:   13.5s
[Parallel(n_jobs=10)]: Done 201 tasks      | elapsed:  

Done in 941.9s
Total ROIs       : 840826
Stenosis    (1)  : 40612
Undetermined(-1) : 50184
Healthy     (0)  : 750030
Saved: C:\Users\ferra\MIC\1r_any_UNICAS\2n_Semestre\Image_Processing_and_Analysis\project\MIC_project\Proposal_StenosisDetection\notebooks\csv_files\oriented_rois\roi_oriented_coordinates.csv


## Extract the features of the oriented ROIs

In [ ]:
# ─────────────────────────────────────────────────────────────
# CELL B: Extract full feature matrix → roi_oriented_features.csv
# Requires coordinates already saved by Cell 2A.
# Only run this when you are ready — it is computationally heavy.
# ─────────────────────────────────────────────────────────────
gabor_bank   = build_gabor_bank(ksize=31, n_orientations=N_ORIENTATIONS, n_sizes=N_SIZES)
feature_cols = build_oriented_feature_column_names(n_filters=len(gabor_bank))

feature_worker_args = [
    (img_path, mask_path, skel_path, gt_boxes,
     gabor_bank, feature_cols, TOTAL_ROIS, RECT_WIDTH, RECT_HEIGHT)
    for img_path, mask_path, skel_path, gt_boxes in full_dataset
]

n_cores = max(1, cpu_count() - 2)
print(f"Extracting features for {len(full_dataset)} images "
      f"using {n_cores} cores...")
start = time.time()

try:
    with Pool(processes=n_cores) as pool:
        feat_results = pool.map(process_one_image_oriented, feature_worker_args)
finally:
    elapsed = time.time() - start
    print(f"Done in {elapsed:.1f}s")

feat_rows = [row for image_rows in feat_results for row in image_rows]
df        = pd.DataFrame(feat_rows)

features_path = OUTPUT_DIR_CSV / "roi_oriented_features.csv"
df.to_csv(features_path, index=False)
print(f"Saved: {features_path}")

In [5]:
# ─────────────────────────────────────────────────────────────
# SENSITIVITY ANALYSIS
# Loads from the lightweight coordinates CSV — no need to
# reload the full feature matrix.
# Only the first TOTAL_ROIS ROIs per image are pipeline-proposed.
# GT-centred ROIs appended afterward are excluded so they
# don't inflate the sensitivity estimate.
# ─────────────────────────────────────────────────────────────
df_coords = pd.read_csv(OUTPUT_DIR_CSV / "roi_oriented_coordinates.csv")

# Extract roi_idx by removing image_name prefix from roi_name
df_coords['roi_idx'] = df_coords.apply(
    lambda row: int(row['roi_name'].replace(row['image_name'] + '_', '')), axis=1
)

rois_per_image = df_coords.groupby('image_name')['roi_idx'].count()
n_less_than_101 = (rois_per_image < 101).sum()
print(f"Images with fewer than 101 ROIs : {n_less_than_101}")
print(f"Images with exactly 101 ROIs    : {(rois_per_image == 101).sum()}")
print(f"Images with more than 101 ROIs  : {(rois_per_image > 101).sum()}")
if n_less_than_101 > 0:
    print(f"\nROI counts for affected images:")
    print(rois_per_image[rois_per_image < 101].to_string())

# Count how many times roi_idx == 101 appears
n_101 = (df_coords['roi_idx'] == 101).sum()
print(f"ROIs with roi_idx == 101 : {n_101}")
print(f"Total unique images      : {df_coords['image_name'].nunique()}")
print(f"Match                    : {n_101 == df_coords['image_name'].nunique()}")

# Exclude the GT-centred ROI (highest roi_idx per image) from sensitivity
max_idx_per_image = df_coords.groupby('image_name')['roi_idx'].transform('max')
df_pipeline = df_coords[df_coords['roi_idx'] != max_idx_per_image].reset_index(drop=True)

pos_per_image    = df_pipeline.groupby('image_name')['label'].apply(lambda x: (x == 1).sum())
n_total_images   = len(pos_per_image)
n_detected       = (pos_per_image >= 1).sum()
sensitivity      = n_detected / n_total_images if n_total_images > 0 else 0.0
avg_pos_per_img  = pos_per_image.mean()
avg_pos_detected = pos_per_image[pos_per_image >= 1].mean()

print("\n" + "═" * 52)
print("  PIPELINE SENSITIVITY REPORT")
print("═" * 52)
print(f"  Total annotated images            : {n_total_images}")
print(f"  Images with ≥1 positive ROI       : {n_detected}")
print(f"  Images with 0 positive ROIs       : {n_total_images - n_detected}")
print(f"  Sensitivity                       : {sensitivity:.4f}  ({sensitivity*100:.2f}%)")
print("─" * 52)
print(f"  Avg positive ROIs / image (all)   : {avg_pos_per_img:.3f}")
print(f"  Avg positive ROIs / image (detected only): {avg_pos_detected:.3f}")
print("═" * 52)

Images with fewer than 101 ROIs : 0
Images with exactly 101 ROIs    : 8324
Images with more than 101 ROIs  : 1
ROIs with roi_idx == 101 : 8325
Total unique images      : 8325
Match                    : True

════════════════════════════════════════════════════
  PIPELINE SENSITIVITY REPORT
════════════════════════════════════════════════════
  Total annotated images            : 8325
  Images with ≥1 positive ROI       : 7788
  Images with 0 positive ROIs       : 537
  Sensitivity                       : 0.9355  (93.55%)
────────────────────────────────────────────────────
  Avg positive ROIs / image (all)   : 3.919
  Avg positive ROIs / image (detected only): 4.190
════════════════════════════════════════════════════


## EXPLORATION OF EXTRACTED FEATURES

In [ ]:
# ─────────────────────────────────────────────────────────────
# EXPLORATION: Load CSVs and reconstruct feature matrix
# ─────────────────────────────────────────────────────────────
df_features_all = pd.read_csv(OUTPUT_DIR_CSV / "roi_oriented_features.csv")

meta_cols    = ['roi_name', 'image_name', 'center_x', 'center_y',
                'angle', 'width', 'height', 'label']
y_all        = df_features_all['label'].values
mask = y_all != -1 # This mask will exclude the -1 labeled ROIs for the exploration. To apply it, just change:
                            # y_all[mask] and X_all[mask]
                   # in the lines below to only use the masked data.

X_all        = df_features_all.drop(columns=meta_cols).values
feature_cols_all = [c for c in df_features_all.columns if c not in meta_cols]

scaler_all   = StandardScaler()
X_scaled_all = scaler_all.fit_transform(X_all)

print(f"Loaded feature matrix : {X_all.shape}")
print(f"Stenosis     (1)      : {np.sum(y_all ==  1)}")
print(f"Undetermined (-1)     : {np.sum(y_all == -1)}")
print(f"Healthy      (0)      : {np.sum(y_all ==  0)}")

In [ ]:
# ─────────────────────────────────────────────────────────────
# EXPLORATION: 3D PCA scatter plot
# ─────────────────────────────────────────────────────────────
%matplotlib tk

pca_3d_all   = PCA(n_components=3)
X_pca_3d_all = pca_3d_all.fit_transform(X_scaled_all)
total_variance_all = pca_3d_all.explained_variance_ratio_.sum() * 100

fig = plt.figure(figsize=(10, 8))
ax  = fig.add_subplot(111, projection='3d')

ax.scatter(X_pca_3d_all[y_all == 0, 0], X_pca_3d_all[y_all == 0, 1], X_pca_3d_all[y_all == 0, 2],
           alpha=0.4, label='Healthy (0)',  color='#3498db', s=25)
ax.scatter(X_pca_3d_all[y_all == 1, 0], X_pca_3d_all[y_all == 1, 1], X_pca_3d_all[y_all == 1, 2],
           alpha=0.8, label='Stenosis (1)', color='#e74c3c', marker='x', s=45)

ax.set_title(f"3D PCA — All Images\n(Total Explained Variance: {total_variance_all:.1f}%)",
             fontsize=14, fontweight='bold', pad=20)
ax.set_xlabel(f"PC 1 ({pca_3d_all.explained_variance_ratio_[0]*100:.1f}%)", fontsize=10, labelpad=10)
ax.set_ylabel(f"PC 2 ({pca_3d_all.explained_variance_ratio_[1]*100:.1f}%)", fontsize=10, labelpad=10)
ax.set_zlabel(f"PC 3 ({pca_3d_all.explained_variance_ratio_[2]*100:.1f}%)", fontsize=10, labelpad=10)
ax.xaxis.set_pane_color((0.95, 0.95, 0.95, 1.0))
ax.yaxis.set_pane_color((0.95, 0.95, 0.95, 1.0))
ax.zaxis.set_pane_color((0.95, 0.95, 0.95, 1.0))
ax.view_init(elev=25, azim=45)
ax.legend(loc='upper left', fontsize=11)
plt.tight_layout()
plt.show()

In [ ]:
# ─────────────────────────────────────────────────────────────
# EXPLORATION: Feature ranking and boxplots
# ─────────────────────────────────────────────────────────────
%matplotlib inline

f_values_all, p_values_all = f_classif(X_scaled_all, y_all)

n_indeces = 8
top_n_indices_all = np.argsort(f_values_all)[::-1][:n_indeces]

print(f"\n--- TOP {n_indeces} MOST DISCRIMINATIVE FEATURES — All Images ---")
for rank, idx in enumerate(top_n_indices_all, start=1):
    print(f"Rank {rank}: Feature {idx:4d}  '{feature_cols_all[idx]}'  — F-score: {f_values_all[idx]:.2f}  p: {p_values_all[idx]:.2e}")

labels_all = np.where(y_all == 0, 'Healthy', 'Stenosis')

fig, axes = plt.subplots(1, n_indeces, figsize=(16, 5))
fig.suptitle(f"Top {n_indeces} Most Discriminative Features — All Images", fontsize=16)

for i, feat_idx in enumerate(top_n_indices_all):
    ax = axes[i]
    sns.boxplot(
        x=labels_all,
        y=X_scaled_all[:, feat_idx],
        hue=labels_all,
        order=['Healthy', 'Stenosis'],
        hue_order=['Healthy', 'Stenosis'],
        palette={'Healthy': '#3498db', 'Stenosis': '#e74c3c'},
        legend=False,
        ax=ax,
    )
    ax.set_title(feature_cols_all[feat_idx], fontsize=7, wrap=True)
    ax.set_xlabel("")
    ax.set_ylabel("Standardised Value" if i == 0 else "")

plt.tight_layout()
plt.show()

In [ ]:
# ─────────────────────────────────────────────────────────────
# EXPLORATION: Scree plot
# ─────────────────────────────────────────────────────────────
%matplotlib inline

pca_full_all = PCA().fit(X_scaled_all)
cumvar_all   = np.cumsum(pca_full_all.explained_variance_ratio_) * 100
n_components_all = np.arange(1, len(cumvar_all) + 1)

thresh_80_all = np.searchsorted(cumvar_all, 80) + 1
thresh_90_all = np.searchsorted(cumvar_all, 90) + 1
thresh_95_all = np.searchsorted(cumvar_all, 95) + 1

fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(n_components_all, cumvar_all, color='#3498db', linewidth=1.5, label='Cumulative explained variance')
ax.fill_between(n_components_all, cumvar_all, alpha=0.1, color='#3498db')

for thresh, n, color in zip([80, 90, 95],
                             [thresh_80_all, thresh_90_all, thresh_95_all],
                             ['#2ecc71', '#e67e22', '#e74c3c']):
    ax.axhline(thresh, color=color, linestyle='--', linewidth=1, alpha=0.7)
    ax.axvline(n,      color=color, linestyle='--', linewidth=1, alpha=0.7)
    ax.annotate(f"{thresh}% → {n} components",
                xy=(n, thresh),
                xytext=(n + len(n_components_all) * 0.02, thresh - 4),
                fontsize=9, color=color)

ax.set_title("Scree Plot — Cumulative Explained Variance (All Images)", fontsize=14, fontweight='bold')
ax.set_xlabel("Number of Components", fontsize=11)
ax.set_ylabel("Cumulative Explained Variance (%)", fontsize=11)
ax.set_xlim(1, len(n_components_all))
ax.set_ylim(0, 101)
ax.grid(True, alpha=0.3)
ax.legend(fontsize=10)
plt.tight_layout()
plt.show()

print(f"Total features      : {X_scaled_all.shape[1]}")
print(f"Components for 80%  : {thresh_80_all}  ({thresh_80_all / X_scaled_all.shape[1] * 100:.1f}% of features)")
print(f"Components for 90%  : {thresh_90_all}  ({thresh_90_all / X_scaled_all.shape[1] * 100:.1f}% of features)")
print(f"Components for 95%  : {thresh_95_all}  ({thresh_95_all / X_scaled_all.shape[1] * 100:.1f}% of features)")